In [ ]:
# Standard library imports
import os
import string
import unicodedata

# SpaCy for NLP tasks
import spacy
from spacy.tokens import Doc, DocBin
from spacy.tokens import Doc, MorphAnalysis

# Parsing CoNLL-U formatted files
from conllu import parse

from pathlib import Path
from typing import List, Tuple, Dict, Union

from sklearn.model_selection import train_test_split

In [ ]:
#model_name = "grc_proiel_trf"
#model_name = "../training/SageMaker/transformer/pipeline_230524_sweep/model-best"
#model_name = '../training/SageMaker/lemmatizer/lemma-trf-26-jan-25-NFC/model-best' # NFC lemmatizer trained on sagemaker 27-1-25
# Check if the model is available and load it
    #if model_name in spacy.info()['pipelines']:
    #    nlp = spacy.load(model_name)
    #else:
    #    raise ImportError(f"The SpaCy model '{model_name}' is not installed.\n"
    #                      "Please install it using 'python -m spacy download {model_name}'.")
nlp = spacy.load('../training/SageMaker/lemmatizer/lemma-trf-26-jan-25-NFC/model-best') # NFC lemmatizer trained on sagemaker 27-1-25


# Load CoNLL-U Data
In this section, we will load and process text data in the CoNLL-U format. CoNLL-U is a standard for annotating text data with linguistic information, such as part-of-speech tags, syntactic dependencies, and morphological features. Our data represents annotated anatomical ancient texts, which is crucial for creating a model that performs well on anatomical texts.

In [ ]:
import unicodedata
import string
from conllu import parse
from typing import List
from sklearn.model_selection import train_test_split
import re

# apostrophes and correct_apostrophe are defined as follows:
apostrophes = ["᾽", "᾿", "'", "’", "‘"]
correct_apostrophe = "ʼ"

def clean_and_remove_accents(text: str) -> str:
    """
    Cleans the given text by removing diacritics (accents), except for specific characters,
    and converting it to lowercase.
    """
    allowed_characters = [' ̓', "᾿", "᾽", "'", "’", "‘", 'ʼ', '̓']  # Including the Greek apostrophe
    if not isinstance(text, str):
        raise ValueError("Input must be a string.")
    try:
        non_accent_chars = [c for c in unicodedata.normalize('NFC', text) 
        if unicodedata.category(c) != 'Mn' or c in allowed_characters]
        return ''.join(non_accent_chars)
    
    except Exception as e:
        # A more generic exception handling if unexpected errors occur
        print(f"An error occurred: {e}")
        return text

In [ ]:
def normalize_text(text: str, form: str = 'NFC', 
                   remove_accents: bool = False, 
                   lowercase: bool = False, 
                   standardize_apostrophe: bool = True, 
                   remove_brackets: bool = False, 
                   remove_trailing_numbers: bool = False, 
                   remove_extra_spaces: bool = False, 
                   debug: bool = False,
                   track_changes: bool = False) -> Union[str, Tuple[str, dict]]:
    """
    Applies multiple text normalization and cleaning steps on the input text.
    If track_changes is True, returns a tuple of (normalized_text, changes_dict)
    """

    normalized_text = text  # Initialize normalized_text with the original text
    changes_dict = {
        'original_text': text,
        'normalized_text': None,  # Will be set at end
        'apostrophe_changes': {},
        'total_apostrophes': {}
    } if track_changes else None
    
    def debug_print(operation_name, before, after):
        if debug:
            print(f"{operation_name} - Before: {before}")
            print(f"{operation_name} - After: {after}")

    if form:
        before_text = normalized_text
        normalized_text = unicodedata.normalize(form, normalized_text)
        debug_print("Unicode normalization", before_text, normalized_text)
            
    if standardize_apostrophe:
        before_text = normalized_text
        if track_changes:
            # Count before changes
            for apos in apostrophes:
                if apos != correct_apostrophe:
                    count = before_text.count(apos)
                    if count > 0:
                        changes_dict['apostrophe_changes'][apos] = count

        # Do the replacement
        apostrophe_map = {ord(apos): ord(correct_apostrophe) for apos in apostrophes}
        normalized_text = normalized_text.translate(apostrophe_map)

        if track_changes:
            # Count correct apostrophes after changes
            correct_count = normalized_text.count(correct_apostrophe)
            if correct_count > 0:
                changes_dict['total_apostrophes'][correct_apostrophe] = correct_count

        debug_print("Standardizing apostrophes", before_text, normalized_text)
        
    if remove_accents:
        before_text = normalized_text
        try:
            normalized_text = clean_and_remove_accents(normalized_text)
        except Exception as e:
            print(f"An error occurred while removing accents: {e}")
            return text        
        debug_print("Removing accents", before_text, normalized_text)
        
    if lowercase:
        before_text = normalized_text
        normalized_text = normalized_text.lower()
        debug_print("Lowercase conversion", before_text, normalized_text)

    if remove_brackets:
        before_text = normalized_text
        normalized_text = re.sub(r'[\(\)\[\]]', '', normalized_text)
        debug_print("Removing brackets", before_text, normalized_text)
        
    if remove_trailing_numbers:
        before_text = normalized_text
        normalized_text = re.sub(r'^\d+|\d+$', '', normalized_text)
        debug_print("Removing trailing numbers", before_text, normalized_text)

    if remove_extra_spaces:
        before_text = normalized_text
        # Split into lines to preserve structure
        lines = normalized_text.split('\n')
        # Process each line separately
        normalized_lines = []
        for line in lines:
            # Split on tabs to preserve CoNLL-U fields
            fields = line.split('\t')
            # Remove extra spaces within each field
            cleaned_fields = [' '.join(field.split()).strip() for field in fields]
            # Put back together with tabs
            normalized_lines.append('\t'.join(cleaned_fields))
        # Put back together with newlines
        normalized_text = '\n'.join(normalized_lines)
        debug_print("Removing extra spaces", before_text, normalized_text)

    if track_changes:
        changes_dict['normalized_text'] = normalized_text
        return normalized_text, changes_dict

    return (normalized_text)

In [ ]:
def remove_accents_to_lowercase(text: str) -> str:
    """
    Cleans the given text by removing diacritics (accents), except for specific characters,
    and converting it to lowercase.
    """
    allowed_characters = [' ̓', "᾿", "᾽", "'", "’", "‘", 'ʼ']  # Including the Greek apostrophe
    if not isinstance(text, str):
        raise ValueError("Input must be a string.")
    try:
        non_accent_chars = [c for c in unicodedata.normalize('NFC', text) 
        if unicodedata.category(c) != 'Mn' or c in allowed_characters]
        return ''.join(non_accent_chars).lower()
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return text

In [ ]:
def normalize_optional_remove_accents(text: str, norm_form: str, apply_cleaning: bool = False) -> str:
    """
    Normalize and optionally clean the given text using specified Unicode normalization form.
    """    
    normalized_text = unicodedata.normalize(norm_form, text)
    if apply_cleaning:
        # Implement the remove_accents_to_lowercase logic to remove unwanted characters or clean text
        text = remove_accents_to_lowercase(normalized_text) # Clean the text
    return text

In [ ]:
#def uniform_apostrophe(token, apostrophes=[' ̓', "᾿", "᾽", "'", "’", "‘"]):
"""Replace specified apostrophes with a uniform representation."""
#return 'ʼ' if token in apostrophes else token

In [ ]:
def process_sentences(sentences_data, nlp, file_name, show_apostrophe_changes=False, debug=False):
    """Converts sentences into spaCy's Doc objects."""
    sentences_data = [t for t in sentences_data if t.get('form')]  # Remove empty tokens

    # Process forms with tracking if needed
    words = [t['form'] for t in sentences_data]
    spaces = [not (t.get('misc') and t['misc'].get('SpaceAfter') == 'No') for t in sentences_data]
    
    # Create Doc object
    doc = Doc(nlp.vocab, words=words, spaces=spaces)
    
    # Set token attributes
    set_token_attributes(doc, sentences_data)
    
    return doc

def set_token_attributes(doc, sentence, debug=False):
    """Set attributes like POS tags and lemmas for each token in the doc."""
    
    EMPTY_VALUES = ['', '_', '—', '-']
    INVALID_POS_VALUES = EMPTY_VALUES + ['X', 'END', 'MID']
    
    # Helper function for debug printing
    def debug_print(message: str) -> None:
        if debug:
            print(message)
    
    # First pass: Set basic attributes and identify root
    root_token = None  # Initialize root token
    
    for i, token in enumerate(doc):
        t = sentence[i]
        
        # Set POS tag based on token type
        if t['form'] in string.punctuation:
            doc[i].pos_ = 'PUNCT'
        elif t.get('upos') not in INVALID_POS_VALUES:
            doc[i].pos_ = t['upos']
        else:
            doc[i].pos_ = ''

        # Set lemma
        doc[i].lemma_ = '' if t.get('lemma') in EMPTY_VALUES else t['lemma']
        
        # Tagger adjustment - POS tags
        # Set tag (xpos)
        if t.get('xpos') and t['xpos'] not in INVALID_POS_VALUES:
            doc[i].tag_ = t['xpos']
        else:
            doc[i].tag_ = ''

        # Set morphological features
        if t.get('feats'):
            morph_analysis_hash = nlp.vocab.morphology.add(t['feats'])
            doc[i].morph = MorphAnalysis.from_id(nlp.vocab, morph_analysis_hash)
        
         # Set dependency relation,performed during Doc creation based on 'dep' and 'head' relation
                # Set dependency relation
        if t.get('deprel') in EMPTY_VALUES:
            doc[i].dep_ = 'None'
        else:
            doc[i].dep_ = t['deprel']
            debug_print(f"dep: {doc[i].dep_}")
            
        # Identify the root token for later use
        if t.get('deprel') == 'root':
            debug_print(f"root token: {root_token}")
            root_token = doc[i]
    
    debug_print(f"root: {root_token if root_token else 'None'}")
    
    # Second pass: Set head relations now that all tokens are processed
    for i, token in enumerate(doc):
        t = sentence[i]
        
        # Set head based on explicit head reference
        if t.get('head') not in [None] + EMPTY_VALUES:
            try:          
                print("head: ", t['head']) if debug else None # Print head index
                head_idx = int(t['head']) - 1  # CoNLL-U uses 1-based indexing
                if 0 <= head_idx < len(doc): # Ensure head index is within bounds
                    debug_print(f"head index: {head_idx}") # Print head index
                    doc[i].head = doc[head_idx] # Assign head token
                                        # Ensure head POS is set
                    if doc[head_idx].pos_:
                        doc[i].head.pos_ = doc[head_idx].pos_

                    debug_print(f"head token: {doc[i].head}")
            except (ValueError, IndexError) as e:
                # Handle invalid head indices
                debug_print(f"Error setting head for token {token.text}: {e}")

        # Fallback to root as head if available and not the current token
        elif root_token and root_token != token:
            doc[i].head = root_token
            debug_print(f"Using root as head: {root_token}")

def clean_and_print_docs(docs, debug=False):
    """Clean and print docs for review."""
    for doc in docs:
        original_sentence = ' '.join(str(doc).split()) # Remove extra spaces
        if debug:
            print(f"Original sentence: {original_sentence}") 
        cleaned_sentence = ' '.join(str(doc).replace('\r', ' ').replace('\n', ' ').split())
        if debug:
            print(f"Cleaned sentence: {cleaned_sentence}")
            if original_sentence != cleaned_sentence:
                print(f"Original sentence: {original_sentence}")
                print(f"Cleaned sentence: {cleaned_sentence}")
            else:
                print(f"Clean sentence: {cleaned_sentence}")
                
def serialize_docs(docs: List[Doc], form: str, output_dir: Path, store_user_data: bool = False, debug: bool = False, output_file_name: str = None):
    output_dir.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists

    # Split documents to subsets
    if docs:  # Check that docs is non-empty to avoid the error
        train_docs, test_docs = train_test_split(docs, test_size=0.2, random_state=42)
        train_docs, dev_docs = train_test_split(train_docs, test_size=0.25, random_state=42)  # Results in 0.2 split for dev
    else:
        print(f"No documents found for normalization form {form}. Skipping serialization.")
        return  # Early return if docs is empty to avoid proceeding with undefined variables

    subsets = {'train': train_docs, 'dev': dev_docs, 'test': test_docs}
    
    for subset_name, subset_docs in subsets.items():
        if output_file_name is not None:
            output_path = output_dir / f"{subset_name}/pos_{subset_name}/{output_file_name}_{subset_name}_{form}.spacy"
        else:
            output_path = output_dir / f"{subset_name}/pos_{subset_name}/pos_{subset_name}_{form}.spacy"
        doc_bin = DocBin(docs=subset_docs, store_user_data=store_user_data)
        doc_bin.to_disk(output_path)
        print(f"Saved {len(subset_docs)} docs for normalization form {form} to {output_path}") # Progress indicator
        
def process_conllu_file(file_path, nlp, normalization_forms, docs_by_norm, apply_cleaning=False, show_apostrophe_changes=False, debug=False):
    file_name = file_path.stem
    print(f"Processing file {file_name}...")  # Progress indicator

    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = f.read()

    # Process each normalization form separately
    for norm in normalization_forms:
        # Apply normalization with correct settings based on apply_cleaning
        if apply_cleaning:
            if show_apostrophe_changes:
                normalized_data, changes = normalize_text(raw_data, 
                                                       form=norm,
                                                       remove_accents=True,
                                                       lowercase=True,
                                                       standardize_apostrophe=True,
                                                       remove_extra_spaces=True,
                                                       debug=debug,
                                                       track_changes=True)
                if debug:
                    print(f"\nFile-level changes for {file_name}:")
                    print(f"Normalization form: {norm}")
                    for apos, count in changes['apostrophe_changes'].items():
                        print(f"Replaced apostrophe: {apos} -> {correct_apostrophe}: {count}")
            else:
                normalized_data = normalize_text(raw_data,
                                              form=norm,
                                              remove_accents=True,
                                              lowercase=False, # Keep case for forms
                                              standardize_apostrophe=True,
                                              remove_extra_spaces=True,
                                              debug=debug)
        else:
            if show_apostrophe_changes:
                normalized_data, changes = normalize_text(raw_data,
                                                       form=norm,
                                                       remove_accents=False,
                                                       lowercase=False, # Keep case for forms
                                                       standardize_apostrophe=True,
                                                       remove_extra_spaces=True,
                                                       debug=debug,
                                                       track_changes=True)
                if debug:
                    print(f"\nFile-level changes for {file_name}:")
                    print(f"Normalization form: {norm}")
                    for apos, count in changes['apostrophe_changes'].items():
                        print(f"Replaced apostrophe: {apos} -> {correct_apostrophe}: {count}")
            else:
                normalized_data = normalize_text(raw_data,
                                              form=norm,
                                              remove_accents=False,
                                              lowercase=False, # Keep case for forms
                                              standardize_apostrophe=False,
                                              remove_extra_spaces=True,
                                              debug=debug)

        print(normalized_data) if debug else None
        
        # Parse the normalized data
        sentences = parse(normalized_data)
        
        for sentence_data in sentences:
            # Before processing, lowercase all lemmas
            for token in sentence_data:
                if token['lemma'] not in ['', '_', '—', '-']:
                    token['lemma'] = token['lemma'].lower()

            # Process sentences without normalization since it's already done
            doc = process_sentences(sentence_data, nlp, 
                                 file_name=file_name,
                                 show_apostrophe_changes=False,
                                 debug=debug)
            docs_by_norm[norm].extend([doc])

def process_folder_and_serialize(input_path, nlp, normalization_forms, output_dir, apply_cleaning=False, show_apostrophe_changes=True, debug: bool = False, output_file_name: str = None):
    docs_by_norm = {norm: [] for norm in normalization_forms}

    # Process each conllu file
    for file_path in Path(input_path).glob("*.conllu"):
        print("file name: ",file_path.name)  # Progress indicator
        process_conllu_file(file_path, nlp, normalization_forms, docs_by_norm, apply_cleaning, show_apostrophe_changes=show_apostrophe_changes, debug=debug)
    
    # Serialize once all documents for a normalization form have been accumulated
    for norm, docs in docs_by_norm.items():
        print(f"Serializing {len(docs)} documents for normalization form {norm} to {output_dir}... for type {type(docs)}")  # Progress indicator
        serialize_docs(docs, norm, output_dir, debug=debug, output_file_name=output_file_name)
                

In [ ]:
# Example usage
process_folder_and_serialize(Path("../assets/INCEpTION_Conllu/"), nlp, ['NFC'], Path("../corpus/"), apply_cleaning=False, show_apostrophe_changes=True, debug=True, output_file_name="pos")



In [ ]:
process_folder_and_serialize(Path("../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFC/"), nlp, ['NFC'], Path("../corpus/"), output_file_name="UD_Ancient_Greek-Perseus", apply_cleaning=False, show_apostrophe_changes=True, debug=True)


In [ ]:
process_folder_and_serialize(Path("../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFC/"), nlp, ['NFC'], Path("../corpus/"), output_file_name="UD_Ancient_Greek-PROIEL", apply_cleaning=False, show_apostrophe_changes=True, debug=True)

In [ ]:
# Tests
import os
from pathlib import Path

# Define the paths to the files
path_to_train_NFKD = Path("../corpus/train/pos_train/pos_train_NFKD.spacy")
path_to_train_NFKC = Path("../corpus/train/pos_train/pos_train_NFKC.spacy")
path_to_dev_NFKD = Path("../corpus/dev/pos_dev/pos_dev_NFKD.spacy")
path_to_dev_NFKC = Path("../corpus/dev/pos_dev/pos_dev_NFKC.spacy")
path_to_test_NFKD = Path("../corpus/test/pos_test/pos_test_NFKD.spacy")
path_to_test_NFKC = Path("../corpus/test/pos_test/pos_test_NFKC.spacy")
path_to_test_NFC = Path("../corpus/test/pos_test/pos_test_NFC.spacy")
path_to_test_NFD = Path("../corpus/test/pos_test/pos_test_NFD.spacy")

# Load the documents
doc_bin_train_NFKD = DocBin().from_disk(path_to_train_NFKD)
doc_bin_train_NFKC = DocBin().from_disk(path_to_train_NFKC)
doc_bin_dev_NFKD = DocBin().from_disk(path_to_dev_NFKD)
doc_bin_dev_NFKC = DocBin().from_disk(path_to_dev_NFKC)
doc_bin_test_NFKD = DocBin().from_disk(path_to_test_NFKD)
doc_bin_test_NFKC = DocBin().from_disk(path_to_test_NFKC)
doc_bin_test_NFC = DocBin().from_disk(path_to_test_NFC)
doc_bin_test_NFD = DocBin().from_disk(path_to_test_NFD)

# Get the documents as a list
docs_train_NFKD = list(doc_bin_train_NFKD.get_docs(nlp.vocab))
docs_train_NFKC = list(doc_bin_train_NFKC.get_docs(nlp.vocab))
docs_dev_NFKD = list(doc_bin_dev_NFKD.get_docs(nlp.vocab))
docs_dev_NFKC = list(doc_bin_dev_NFKC.get_docs(nlp.vocab))
docs_test_NFKD = list(doc_bin_test_NFKD.get_docs(nlp.vocab))
docs_test_NFKC = list(doc_bin_test_NFKC.get_docs(nlp.vocab))
docs_test_NFC = list(doc_bin_test_NFC.get_docs(nlp.vocab))
docs_test_NFD = list(doc_bin_test_NFD.get_docs(nlp.vocab))

In [ ]:
# Tests
import os
from pathlib import Path

# Define the paths to the files
path_to_train_NFKD = Path("../corpus/train/pos_train/UD_Ancient_Greek-PROIEL_train_NFKD.spacy")
path_to_train_NFKC = Path("../corpus/train/pos_train/UD_Ancient_Greek-PROIEL_train_NFKC.spacy")
path_to_train_NFD = Path("../corpus/train/pos_train/UD_Ancient_Greek-PROIEL_train_NFD.spacy")
path_to_train_NFC = Path("../corpus/train/pos_train/UD_Ancient_Greek-PROIEL_train_NFC.spacy")

path_to_dev_NFKD = Path("../corpus/dev/pos_dev/UD_Ancient_Greek-PROIEL_dev_NFKD.spacy")
path_to_dev_NFKC = Path("../corpus/dev/pos_dev/UD_Ancient_Greek-PROIEL_dev_NFKC.spacy")
path_to_dev_NFD = Path("../corpus/dev/pos_dev/UD_Ancient_Greek-PROIEL_dev_NFD.spacy")
path_to_dev_NFC = Path("../corpus/dev/pos_dev/UD_Ancient_Greek-PROIEL_dev_NFC.spacy")

path_to_test_NFKD = Path("../corpus/test/pos_test/UD_Ancient_Greek-PROIEL_test_NFKD.spacy")
path_to_test_NFKC = Path("../corpus/test/pos_test/UD_Ancient_Greek-PROIEL_test_NFKC.spacy")
path_to_test_NFD = Path("../corpus/test/pos_test/UD_Ancient_Greek-PROIEL_test_NFD.spacy")
path_to_test_NFC = Path("../corpus/test/pos_test/UD_Ancient_Greek-PROIEL_test_NFC.spacy")

# Load the documents
doc_bin_train_NFKD = DocBin().from_disk(path_to_train_NFKD)
doc_bin_train_NFKC = DocBin().from_disk(path_to_train_NFKC)
doc_bin_train_NFD = DocBin().from_disk(path_to_train_NFD)
doc_bin_train_NFC = DocBin().from_disk(path_to_train_NFC)

doc_bin_dev_NFKD = DocBin().from_disk(path_to_dev_NFKD)
doc_bin_dev_NFKC = DocBin().from_disk(path_to_dev_NFKC)
doc_bin_dev_NFD = DocBin().from_disk(path_to_dev_NFD)
doc_bin_dev_NFC = DocBin().from_disk(path_to_dev_NFC)

doc_bin_test_NFKD = DocBin().from_disk(path_to_test_NFKD)
doc_bin_test_NFKC = DocBin().from_disk(path_to_test_NFKC)
doc_bin_test_NFC = DocBin().from_disk(path_to_test_NFC)
doc_bin_test_NFD = DocBin().from_disk(path_to_test_NFD)

# Get the documents as a list
docs_train_NFKD = list(doc_bin_train_NFKD.get_docs(nlp.vocab))
docs_train_NFKC = list(doc_bin_train_NFKC.get_docs(nlp.vocab))
docs_train_NFD = list(doc_bin_train_NFD.get_docs(nlp.vocab))
docs_train_NFC = list(doc_bin_train_NFC.get_docs(nlp.vocab))

docs_dev_NFKD = list(doc_bin_dev_NFKD.get_docs(nlp.vocab))
docs_dev_NFKC = list(doc_bin_dev_NFKC.get_docs(nlp.vocab))
docs_dev_NFD = list(doc_bin_dev_NFD.get_docs(nlp.vocab))
docs_dev_NFC = list(doc_bin_dev_NFC.get_docs(nlp.vocab))

docs_test_NFKD = list(doc_bin_test_NFKD.get_docs(nlp.vocab))
docs_test_NFKC = list(doc_bin_test_NFKC.get_docs(nlp.vocab))
docs_test_NFC = list(doc_bin_test_NFC.get_docs(nlp.vocab))
docs_test_NFD = list(doc_bin_test_NFD.get_docs(nlp.vocab))

In [ ]:
# Search for specific text in documents
sample_text = "ταῦτα δὲ τὰ γεγραμμένα πάσιν ὁμοίως εἰσί, καὶ φλέβες αἱ γεγραμμένοι"

def find_text_in_docs(docs, doc_type):
    for doc in docs:
        if sample_text in doc.text:
            print(f"Found in {doc_type}")
            print(doc.text)
            for token in doc:
                print("token: ", token.text, "LEMMA: ", token.lemma_, "POS: ", token.pos_, "TAG: ", token.tag_, "DEP: ", token.dep_, "HEAD: ",token.head, "Head POS: ", token.head.pos_,
                      "Children: ",[child for child in token.children])
        else:
            continue  # stop after finding the first match

find_text_in_docs(docs_train_NFKD, "train NFKD")
find_text_in_docs(docs_train_NFKC, "train NFKC")
find_text_in_docs(docs_dev_NFKD, "dev NFKD")
find_text_in_docs(docs_dev_NFKC, "dev NFKC")
find_text_in_docs(docs_test_NFKD, "test NFKD")
find_text_in_docs(docs_test_NFKC, "test NFKC")
find_text_in_docs(docs_test_NFKC, "test NFC")
find_text_in_docs(docs_test_NFKC, "test NFD")



In [ ]:
# Print tokens with non-empty POS tags
for doc in docs_test_NFD:
    for token in doc:
        if token.pos_ != '':
            print("token: ", token.text, "LEMMA: ", token.lemma_, "POS: ", token.pos_, "TAG: ", token.tag_, "DEP: ", token.dep_, "HEAD: ", token.head, "Head POS: ", token.head.pos_,
                "Children: ", [child for child in token.children])

In [ ]:
def compare_encoding_norms(docs_list, doc_type_list):
    """
    Compare the size and character count of documents in different encoding norms.
    """
    for docs, doc_type in zip(docs_list, doc_type_list):
        total_size = sum(len(doc.text) for doc in docs)
        total_chars = sum(len(doc.text.replace(' ', '')) for doc in docs)
        print(f'For {doc_type}: Total size = {total_size}, Total characters = {total_chars}')

# Compare encoding norms
compare_encoding_norms([
    docs_test_NFKD, docs_test_NFKC, docs_test_NFC, docs_test_NFD
], [
    'test NFKD', 'test NFKC', 'test NFC', 'test NFD'
])

In [ ]:
def count_lemmas(docs_list, doc_type_list):
    """
    Count the total number of lemmas, the number of unique lemmas, and the number of unique lemmas with lowercase characters in each file.
    """
    for docs, doc_type in zip(docs_list, doc_type_list):
        all_lemmas = []
        all_lemmas_lower = []
        for doc in docs:
            for token in doc:
                if token.lemma_:
                    all_lemmas.append(token.lemma_)
                    all_lemmas_lower.append(token.lemma_.lower())
        unique_lemmas = set(all_lemmas)
        unique_lemmas_lower = set(all_lemmas_lower)
        print(f'For {doc_type}: Total lemmas = {len(all_lemmas)}, Unique lemmas = {len(unique_lemmas)}, Unique lemmas (lowercase) = {len(unique_lemmas_lower)}')

# Count lemmas
count_lemmas([
    docs_train_NFKD, docs_train_NFKC, docs_train_NFD, docs_train_NFC,
    docs_dev_NFKD, docs_dev_NFKC, docs_dev_NFD, docs_dev_NFC,
    docs_test_NFKD, docs_test_NFKC, docs_test_NFD, docs_test_NFC
], [
    'test NFKD', 'test NFKC', 'test NFC', 'test NFD'
])


In [ ]:
def find_specific_lemma(docs_list, doc_type_list, lemma):
    """
    Find a specific lemma in the documents and also look for it in lowercase, then print the documents where it is found.
    """
    # Convert the lemma to lowercase once to avoid repeated conversions
    lemma_lower = lemma.lower()
    
    for docs, doc_type in zip(docs_list, doc_type_list):
        for doc in docs:
            for token in doc:
                if token.lemma_ == lemma or token.lemma_ == lemma_lower:
                    print(f'Found lemma "{token.lemma_}" in {doc_type}: {doc.text}')
                    break  # Stop searching in this document once the lemma is found

# Find specific lemma
lemma_to_find = "example_lemma"  # Replace with the lemma you want to find
find_specific_lemma([
    docs_train_NFKD, docs_train_NFKC, docs_train_NFD, docs_train_NFC,
    docs_dev_NFKD, docs_dev_NFKC, docs_dev_NFD, docs_dev_NFC,
    docs_test_NFKD, docs_test_NFKC, docs_test_NFD, docs_test_NFC], [
    'test NFKD', 'test NFKC', 'test NFC', 'test NFD'
], 'Ἰησοῦς')

In [ ]:
#make word lowercase
word = 'εὐαγγελιζόμενος'
lower_word= word.lower()
print(lower_word)